# Tool Schema Design: the Agent-Computer Interface

**Phase 03** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-03/02-tool-schema-design.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic pydantic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 03-02: Tool Schema Design
Demonstrate schema quality progression (bad -> better -> good) and tool call validation.

Run: python main.py
Run validation demo: python main.py --validate
Run Pydantic demo:   python main.py --pydantic
"""

import argparse
import json
from typing import Any, Optional

import anthropic

### Three versions of the same tool schema (bad -> better -> good)

In [ ]:
# Version 1: Bad - abbreviated names, no descriptions, all required, no constraints
SCHEMA_V1 = {
    "name": "search",
    "description": "search products",
    "input_schema": {
        "type": "object",
        "properties": {
            "q":    {"type": "string"},
            "n":    {"type": "integer"},
            "sort": {"type": "string"},
            "f":    {"type": "string"},
        },
        "required": ["q", "n", "sort", "f"],
    },
}

# Version 2: Better - natural names, basic descriptions, fewer required
SCHEMA_V2 = {
    "name": "search_products",
    "description": "Search the product catalog.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query.",
            },
            "max_results": {
                "type": "integer",
                "description": "Maximum number of results. Default 10.",
            },
            "sort_by": {
                "type": "string",
                "description": "Sort order. Options: relevance, price_asc, price_desc, newest.",
            },
            "filters": {
                "type": "object",
                "description": "Optional filters.",
            },
        },
        "required": ["query"],
    },
}

# Version 3: Good - enum constraints, examples in descriptions, sensible defaults
SCHEMA_V3 = {
    "name": "search_products",
    "description": (
        "Search the product catalog by keyword. "
        "Use this when the user wants to find, browse, or look up products. "
        "Do not use for order lookups or account information."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": (
                    "Natural-language search query. "
                    "Examples: 'blue running shoes', 'waterproof jacket under $200', 'size 10 boots'."
                ),
            },
            "max_results": {
                "type": "integer",
                "description": "Maximum items to return. Default: 10. Range: 1 to 50.",
            },
            "sort_by": {
                "type": "string",
                "enum": ["relevance", "price_asc", "price_desc", "newest"],
                "description": (
                    "Sort order. Default: 'relevance'. "
                    "Use 'price_asc' for cheapest-first, 'price_desc' for most-expensive-first, "
                    "'newest' for recently added items."
                ),
            },
            "filters": {
                "type": "object",
                "description": (
                    "Optional filter criteria. All keys optional. "
                    "Example: {\"min_price\": 20, \"max_price\": 150, \"category\": \"footwear\", \"in_stock\": true}."
                ),
                "properties": {
                    "min_price": {"type": "number",  "description": "Minimum price in USD."},
                    "max_price": {"type": "number",  "description": "Maximum price in USD."},
                    "category":  {"type": "string",  "description": "Product category, e.g. 'footwear'."},
                    "in_stock":  {"type": "boolean", "description": "If true, return only in-stock items."},
                },
            },
        },
        "required": ["query"],
    },
}

### Stub function

In [ ]:
def search_products(
    query: str,
    max_results: int = 10,
    sort_by: str = "relevance",
    filters: Optional[dict] = None,
) -> dict:
    """Stub: returns fake product search results."""
    products = [
        {"id": "P001", "name": "Blue Trail Runners", "price": 89.99,  "in_stock": True,  "category": "footwear"},
        {"id": "P002", "name": "Blue Road Runners",  "price": 119.99, "in_stock": True,  "category": "footwear"},
        {"id": "P003", "name": "Blue Casual Sneakers","price": 59.99, "in_stock": False, "category": "footwear"},
        {"id": "P004", "name": "Waterproof Jacket",   "price": 179.99,"in_stock": True,  "category": "outerwear"},
        {"id": "P005", "name": "Running Socks 3-pack","price": 14.99, "in_stock": True,  "category": "accessories"},
    ]

    # Apply filters if provided
    if filters:
        if filters.get("min_price"):
            products = [p for p in products if p["price"] >= filters["min_price"]]
        if filters.get("max_price"):
            products = [p for p in products if p["price"] <= filters["max_price"]]
        if filters.get("category"):
            products = [p for p in products if p["category"] == filters["category"]]
        if filters.get("in_stock"):
            products = [p for p in products if p["in_stock"]]

    # Apply sort
    sort_key = {"relevance": None, "price_asc": lambda x: x["price"],
                "price_desc": lambda x: -x["price"], "newest": None}.get(sort_by)
    if sort_key:
        products = sorted(products, key=sort_key)

    return {
        "query": query,
        "total": len(products),
        "results": products[:max_results],
        "sort_by": sort_by,
    }

### Tool call validator

In [ ]:
def validate_tool_call(tool_input: dict, schema: dict) -> list[str]:
    """
    Validates a tool_input dict against the input_schema from a tool definition.
    Returns a list of error strings. Empty list means the call is valid.
    """
    errors: list[str] = []
    input_schema = schema.get("input_schema", {})
    properties = input_schema.get("properties", {})
    required_fields = input_schema.get("required", [])

    for field in required_fields:
        if field not in tool_input:
            errors.append(f"Missing required field: '{field}'")

    type_map = {
        "string":  str,
        "integer": int,
        "number":  (int, float),
        "boolean": bool,
        "object":  dict,
        "array":   list,
    }

    for field, value in tool_input.items():
        if field not in properties:
            errors.append(f"Unknown field: '{field}'")
            continue

        prop_schema = properties[field]
        expected_type = prop_schema.get("type")

        if expected_type in type_map:
            expected_python_type = type_map[expected_type]
            if not isinstance(value, expected_python_type):
                errors.append(
                    f"Field '{field}': expected {expected_type}, "
                    f"got {type(value).__name__} ({value!r})"
                )

        if "enum" in prop_schema and value not in prop_schema["enum"]:
            errors.append(
                f"Field '{field}': value {value!r} not in allowed values {prop_schema['enum']}"
            )

    return errors

### Schema comparison demo (no API call)

In [ ]:
def run_validation_demo() -> None:
    """Show how the validator catches errors from bad-schema-style calls."""
    test_calls = [
        {
            "label": "V1-style call (abbreviated names, wrong types)",
            "input": {"q": "blue shoes", "n": 10, "sort": "newest", "f": "{}"},
        },
        {
            "label": "V2-style call (right names but invalid sort_by)",
            "input": {"query": "blue shoes", "max_results": 10, "sort_by": "ascending"},
        },
        {
            "label": "V3-style call (correct)",
            "input": {"query": "blue running shoes", "max_results": 10, "sort_by": "price_asc"},
        },
        {
            "label": "Missing required field",
            "input": {"max_results": 5, "sort_by": "newest"},
        },
        {
            "label": "Wrong type for max_results",
            "input": {"query": "shoes", "max_results": "ten"},
        },
    ]

    print("\n=== Tool Call Validation Demo ===\n")
    print(f"Schema under test: {SCHEMA_V3['name']}")
    print()

    for test in test_calls:
        errors = validate_tool_call(test["input"], SCHEMA_V3)
        status = "VALID" if not errors else f"INVALID ({len(errors)} error(s))"
        print(f"Call: {test['label']}")
        print(f"  Input:  {json.dumps(test['input'])}")
        print(f"  Status: {status}")
        for e in errors:
            print(f"    - {e}")
        print()

### Pydantic schema generation demo

In [ ]:
def run_pydantic_demo() -> None:
    """Show Pydantic-generated schema vs hand-written V3 schema."""
    try:
        from pydantic import BaseModel, Field
    except ImportError:
        print("Install pydantic: pip install pydantic")
        return

    class ProductFilters(BaseModel):
        min_price: Optional[float] = Field(None, description="Minimum price in USD.")
        max_price: Optional[float] = Field(None, description="Maximum price in USD.")
        category:  Optional[str]   = Field(None, description="Product category, e.g. 'footwear'.")
        in_stock:  Optional[bool]  = Field(None, description="If true, return only in-stock items.")

    class SearchProductsInput(BaseModel):
        query: str = Field(
            description=(
                "Natural-language search query. "
                "Examples: 'blue running shoes', 'waterproof jacket under $200'."
            )
        )
        max_results: int = Field(
            default=10, ge=1, le=50,
            description="Maximum items to return. Default: 10. Range: 1 to 50."
        )
        sort_by: str = Field(
            default="relevance",
            description="Sort order. One of: relevance, price_asc, price_desc, newest."
        )
        filters: Optional[ProductFilters] = Field(
            None,
            description=(
                "Optional filters. Example: {min_price: 20, max_price: 150, category: 'footwear'}."
            )
        )

    schema = SearchProductsInput.model_json_schema()
    schema.pop("title", None)
    tool_schema = {
        "name": "search_products",
        "description": (
            "Search the product catalog by keyword. "
            "Use when the user wants to find, browse, or look up products."
        ),
        "input_schema": schema,
    }

    print("\n=== Pydantic-Generated Schema ===")
    print(json.dumps(tool_schema, indent=2))

    # Validate a call using Pydantic directly
    print("\n=== Pydantic Validation ===")
    good_input = {"query": "blue shoes", "max_results": 10, "sort_by": "price_asc"}
    bad_input  = {"query": "blue shoes", "max_results": 100}  # exceeds le=50

    try:
        validated = SearchProductsInput.model_validate(good_input)
        print(f"Good input: VALID -> {validated.model_dump()}")
    except Exception as e:
        print(f"Good input: INVALID -> {e}")

    try:
        validated = SearchProductsInput.model_validate(bad_input)
        print(f"Bad input:  VALID -> {validated.model_dump()}")
    except Exception as e:
        print(f"Bad input:  INVALID -> {e}")

### Live schema comparison via API

In [ ]:
def run_schema_comparison(user_message: str) -> None:
    """
    Call the LLM with V1 and V3 schemas on the same message.
    Print what tool_input the LLM generates for each.
    """
    client = anthropic.Anthropic()

    print(f"\n[user] {user_message}")
    print()

    for label, schema in [("V1 (bad schema)", SCHEMA_V1), ("V3 (good schema)", SCHEMA_V3)]:
        response = client.messages.create(
            model="claude-3-5-haiku-20241022",
            max_tokens=256,
            tools=[schema],
            messages=[{"role": "user", "content": user_message}],
        )

        tool_uses = [b for b in response.content if b.type == "tool_use"]
        if tool_uses:
            tu = tool_uses[0]
            errors = validate_tool_call(tu.input, schema)
            status = "VALID" if not errors else f"INVALID ({len(errors)} error)"
            print(f"  {label}")
            print(f"    Tool called: {tu.name}")
            print(f"    Input:       {json.dumps(tu.input)}")
            print(f"    Validation:  {status}")
            for e in errors:
                print(f"      - {e}")
        else:
            text = next((b.text for b in response.content if hasattr(b, "text")), "no text")
            print(f"  {label}: No tool call. Response: {text[:80]}")
        print()

### Main

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(description="Lesson 03-02: Tool Schema Design")
    parser.add_argument("--validate", action="store_true", help="Run validation demo (no API call)")
    parser.add_argument("--pydantic", action="store_true", help="Run Pydantic schema demo (no API call)")
    parser.add_argument(
        "--compare",
        action="store_true",
        help="Compare V1 vs V3 schema via live API call",
    )
    parser.add_argument(
        "--message",
        default="Find blue running shoes under $100, sorted by price.",
        help="User message for API comparison demo.",
    )
    args = parser.parse_args()

    if args.validate:
        run_validation_demo()
        return

    if args.pydantic:
        run_pydantic_demo()
        return

    if args.compare:
        print("=== 03-02: Schema Comparison (V1 vs V3) ===")
        run_schema_comparison(args.message)
        return

    # Default: run validation demo + show schema diff summary
    print("=== 03-02: Tool Schema Design ===")
    print("\nRunning validation demo (no API call required)...")
    run_validation_demo()
    print("\nTo compare V1 vs V3 via live API: python main.py --compare")
    print("To see Pydantic schema generation:  python main.py --pydantic")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. Which two schema design rules are violated here, and which fix has the highest impact?**

- A. Rules 1 and 4: the tool does too many things (one tool, one purpose) and the string field needs an enum. Splitting into separate tools has the highest impact because each tool gets a clear, specific description.
- B. Rules 2 and 5: parameter names are ambiguous and descriptions lack examples. Add examples to the description.
- C. Rules 3 and 4: request_type should be optional with a default, and an enum should be added. Making request_type optional is the highest-impact fix.
- D. Rules 1 and 5: the tool name is vague and descriptions lack examples. Renaming the tool has the highest impact.

**2. What is the root cause, and what is the correct fix?**

- A. The LLM is choosing random parameter names; fix by adding a strict JSON schema validator in the dispatch layer
- B. 'uid' is an abbreviation that the LLM doesn't recognize as a convention; rename the parameter to 'user_id' which matches natural language and common API conventions
- C. The description 'user id' should say 'uid: the user id field'; adding the parameter name to the description helps
- D. Add all three variants (uid, user_id, userId) as accepted parameter names in the dispatch layer

**3. What does this distribution tell you about schema design, and what is the correct fix?**

- A. The LLM is unpredictable and cannot reliably use string parameters; replace with a boolean `ascending: bool`
- B. The LLM generates natural-language variants when no enum constrains the values; add `enum: ['asc', 'desc']` and update the description to show 'asc' and 'desc' as the exact expected values
- C. Add a normalization function in the dispatch layer that maps all variants to 'asc' or 'desc'
- D. The description should list all accepted variants so the LLM knows which ones work

**4. What schema design rule is violated, and what is the fix?**

- A. Rule 5: descriptions are missing; add descriptions so the LLM knows what to put in each field
- B. Rule 3: fields the LLM cannot infer from the user's message should be optional with documented defaults, not required
- C. Rule 1: this tool is too complex; split it into create_ticket_basic and create_ticket_full
- D. Rule 4: priority and tags need enum constraints to reduce validation failures

**5. What does this reveal about the value of the validator, and at what layer should validation happen?**

- A. The validator is unnecessary because JSON types are always correct when parsed from the LLM's output
- B. The validator catches type coercion failures that would otherwise cause silent wrong behavior; validation should happen at the dispatch layer before the function is called, not at the API layer
- C. The fix is to make the search API handle both strings and integers; the schema should not enforce strict types
- D. The LLM should be prompted to always use integer literals; prompt engineering removes the need for validation

**6. What is the most precise argument for fixing HIGH severity findings before shipping?**

- A. HIGH severity findings are defined as issues that cause incorrect or failed tool calls in production, not style preferences; shipping them guarantees production failures
- B. Fixing schemas after shipping requires a redeployment; it's always cheaper to fix before launch
- C. The prompt may be wrong; HIGH findings should be reviewed by a human before treating them as blockers
- D. HIGH severity findings also affect cost; bad schemas generate more tokens and more retries

**7. What is the schema fix, and which rule does it invoke?**

- A. Rule 1: merge the two tools into one `search(source, query)` tool to avoid ambiguity
- B. Rule 5: both tools need descriptions that explain when to use each one specifically, with examples of the type of query each handles
- C. Rule 2: rename the parameters to `kb_query` and `web_query` to distinguish them
- D. Rule 3: make `query` optional in one tool so the LLM can distinguish them by required fields

**8. What is wrong with this reasoning?**

- A. Pydantic-generated schemas never include descriptions; you must always write them manually in a separate JSON file
- B. Pydantic automates schema generation but the field names are abbreviations that violate Rule 2; and 'self-explanatory' to the engineer does not mean the LLM will use them correctly - descriptions with examples are required
- C. The schema generation is correct; the problem is that Pydantic's JSON Schema format is not compatible with the Anthropic API
- D. Self-explanatory names are fine for simple fields; only complex nested objects need descriptions

_Answers are in `checks.json` in the lesson directory._